# Design an Incremental Ingestion Strategy — Solution


## Initial Load Plan

**Strategy:** Parallel full extraction by entity, respecting 5,000 req/hr rate limit.

| Entity | Records | Pages (100/page) | API Calls |
|--------|---------|-------------------|-----------|
| Projects | 2,000 | 20 | 20 |
| Tasks | 150,000 | 1,500 | 1,500 |
| Comments | 500,000 | 5,000 | 5,000 |
| Users | 800 | 8 | 8 |
| **Total** | | | **6,528** |

**Time estimate:** 6,528 requests ÷ 5,000/hr = ~1.3 hours (plus OAuth refreshes every 60 min = 2 token calls).

**Execution plan:**
1. Extract Users first (8 calls, needed as reference)
2. Extract Projects (20 calls)
3. Extract Tasks + Comments in parallel (2 Step Functions branches: Tasks gets 2,500 req/hr budget, Comments gets 2,500 req/hr budget)
4. Total: ~1.5 hours including buffer for retries

## Incremental Strategy

| Attribute | Value |
|-----------|-------|
| **Change detection** | `updated_at` timestamp field (all entities support it) |
| **Sync frequency** | Every 4 hours (EventBridge Scheduler) |
| **High-water mark** | Stored in DynamoDB table `sync_state` — one row per entity with `last_sync_timestamp` |
| **Query** | `GET /api/v1/{entity}?updated_after={high_water_mark}&page_size=100&sort=updated_at:asc` |
| **Daily volume** | ~5,000 tasks + ~15,000 comments = ~200 pages/run = well within rate limit |
| **Soft deletes** | Query `GET /api/v1/{entity}?deleted_after={high_water_mark}` separately; write to bronze with `_deleted_at` column; Silver MERGE sets `is_deleted = true` (no physical delete) |
| **Idempotency** | Each sync writes to a date-partitioned bronze prefix with `batch_id`; re-runs overwrite same partition via Iceberg |

## Error Handling Matrix

| Scenario | Response | Recovery |
|----------|----------|----------|
| 429 (rate limited) | Respect `Retry-After` header; Step Functions Wait state pauses execution | Automatic resume after wait; no data loss |
| 500 on page 47 of 150 | Log failed page; retry 3x with exponential backoff (2s, 8s, 32s) | If still failing after 3 retries, checkpoint at page 46; alert via SNS; next run resumes from page 47 |
| OAuth token expires mid-extraction | Catch 401 response; refresh token from Secrets Manager; retry the failed request | Automatic; Step Functions has a token-refresh state that triggers on auth failure |
| Record fails schema validation | Write to quarantine prefix (`bronze/quarantine/{entity}/{date}/`); continue processing remaining records | Alert pipeline owner; quarantined records reviewed manually; don't block pipeline |

## Schema Evolution Approach

**When the vendor adds a new field to Tasks:**

1. **Bronze layer:** No action needed — raw JSON is stored as-is. New fields appear automatically.
2. **Detection:** Glue Crawler (scheduled weekly) detects schema drift and logs new columns.
3. **Silver layer:** Iceberg schema evolution adds new nullable column automatically (`ALTER TABLE ... ADD COLUMN`). No data rewrite required.
4. **Notification:** CloudWatch alarm triggers when crawler detects new fields → SNS → domain team reviews whether the field should propagate to Gold.
5. **Breaking changes (field removal/rename):** Detected by comparing API response keys against expected schema. Records with missing required fields → quarantine. Pipeline owner alerted to update mapping.

## Mermaid Sync Flow Diagram

```mermaid
sequenceDiagram
    participant EB as EventBridge (4-hourly)
    participant SF as Step Functions
    participant DDB as DynamoDB (state)
    participant API as SaaS API
    participant S3 as S3 Bronze
    participant Glue as Glue ETL

    EB->>SF: Trigger sync workflow
    SF->>DDB: Read high-water mark
    SF->>API: Refresh OAuth token
    loop For each entity (parallel)
        SF->>API: GET /entity?updated_after={hwm}&page=1
        API-->>SF: Page of records
        Note over SF,API: Paginate until empty response
        SF->>S3: Write batch to bronze/{entity}/{date}/
    end
    SF->>DDB: Update high-water mark
    SF->>Glue: Trigger bronze→silver ETL
    Note over Glue: Deduplicate, validate, merge into silver
```